# 1. Poll Data Exploration & Baseline

This notebook loads the 2022 Colombian presidential poll data, visualizes it, and computes a baseline weighted average. The baseline serves as a benchmark: the Bayesian model (notebooks 2-3) should improve on this.

**References**: `docs/data/polls.md` | `docs/architecture/features.md` §6

In [ ]:
import os, sys
# Ensure CWD is the project root (parent of notebooks/)
if os.path.basename(os.getcwd()) in ("notebooks", ""):
    os.chdir("..")
sys.path.insert(0, ".")


In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd, numpy as np, matplotlib.pyplot as plt

from co_president.config import (
    ModelConfig, ELECTION_DATE_ROUND1, ELECTION_DATE_ROUND2,
    POLLSTER_RATINGS, FIRST_ROUND_CANDIDATES
)
from co_president.data import load_and_clean_all, load_canonical_results
from co_president.aggregation import (
    combined_weight, weighted_average, evolution_series
)

plt.rcParams['figure.dpi'] = 100


## Load & Inspect Data

In [ ]:
cp = load_and_clean_all()
r1, r2 = load_canonical_results()

print(f"R1 polls: {len(cp.round1)}")
print(f"R2 polls: {len(cp.round2)}")
print(f"R1 pollsters: {cp.round1['encuestadora'].nunique()}")
print(f"R2 pollsters: {cp.round2['encuestadora'].nunique()}")
cp.round1[['fecha','encuestadora','muestra','gustavo_petro','rodolfo_hernandez']].head()


## Poll Time Series

Plot candidate shares over time. Marker size scales with poll sample size.

In [ ]:
candidates = ['gustavo_petro','rodolfo_hernandez','federico_gutierrez','sergio_fajardo']
fig, ax = plt.subplots(figsize=(12, 5))
for c in candidates:
    subset = cp.round1.dropna(subset=[c])
    ax.scatter(subset['fecha'], subset[c], s=subset['muestra']/50,
               alpha=0.5, label=c.replace('_',' ').title())
    ax.axhline(r1.get_share(c)*100, linestyle='--', color='gray', alpha=0.3)
ax.set_ylabel('Share (%)')
ax.set_title('R1 Polling Time Series')
ax.legend()
plt.show()


## Baseline Weighted Average

The `weighted_average` function computes: `w = w_time × w_sample × w_pollster`, normalized to sum=1.

In [ ]:
candidate_keys = sorted(set(FIRST_ROUND_CANDIDATES.keys()) & set(cp.round1.columns))
baseline = weighted_average(cp.round1, candidate_keys, ELECTION_DATE_ROUND1, POLLSTER_RATINGS)
print("Candidate          | Predicted | Actual  | Error")
print("-" * 55)
for ck in candidate_keys:
    pred = baseline[ck]
    actual = r1.get_share(ck) * 100
    print(f"{ck:20s} | {pred:7.2f}%  | {actual:6.2f}% | {pred-actual:+6.2f}pp")
mae = sum(abs(baseline[ck] - r1.get_share(ck)*100) for ck in candidate_keys) / len(candidate_keys)
print(f"\nBaseline MAE: {mae:.2f}pp")


## Evolution Series

How did the weighted average evolve over the campaign?

In [ ]:
evol = evolution_series(cp.round1, candidate_keys, ELECTION_DATE_ROUND1, POLLSTER_RATINGS, n_snapshots=20)
fig, ax = plt.subplots(figsize=(12, 5))
for ck in candidate_keys:
    subset = evol[evol['candidate']==ck]
    ax.plot(subset['as_of_date'], subset['weighted_average'], label=ck.replace('_',' ').title())
    ax.axhline(r1.get_share(ck)*100, linestyle='--', color='gray', alpha=0.3)
ax.set_ylabel('Weighted average (%)')
ax.set_title('Evolution of Weighted Polling Averages')
ax.legend()
plt.show()


## Summary

- The baseline weighted average gives a first-pass estimate.
- The Bayesian model (Notebook 2) should improve on this by:
  - Learning time dynamics via the random walk
  - Correcting for pollster bias via house effects
  - Incorporating the actual election result as a constraint